# Khảo sát dữ liệu bệnh nhân suy tim
Võ Minh Huy - 2311110117. Kiểm tra original và before; chưa tạo after. Số dòng CSV tính cả hàng tiêu đề ở dòng 1.

In [1]:
from pathlib import Path
import importlib.util
import pandas as pd
root = Path.cwd()
if not (root / 'DanYMonHoc.pdf').exists():
    root = root.parent
spec = importlib.util.spec_from_file_location('audit', root / 'python' / '01_kiem_tra_du_lieu.py')
audit = importlib.util.module_from_spec(spec)
spec.loader.exec_module(audit)
summary = audit.run()

{
  "original_rows": 299,
  "before_rows": 309,
  "columns": 13,
  "empty_rows": 5,
  "nonempty_rows": 304,
  "exact_duplicate_extra_rows": 3,
  "normalized_duplicate_extra_rows": 5,
  "appended_patient_rows": 5,
  "empty_csv_lines": [
    306,
    307,
    308,
    309,
    310
  ],
  "changed_cells_299_including_formatting": 749,
  "numeric_format_only_cells": 582,
  "substantive_changed_cells_299": 167,
  "missing_cells_first299": 120,
  "missing_cells_nonempty_before": 122,
  "change_type_counts_first299": {
    "numeric_format_only": 582,
    "missing": 120,
    "category_value_changed": 2,
    "numeric_value_changed": 5,
    "category_representation": 38,
    "wrong_type": 1,
    "invalid_target": 1
  },
  "input_sha256": {
    "heart_failure_clinical_records_dataset_original.csv": "9c73cea7468ff5d517801ec050fe9993da5912fce4b56f296f8df3b38dd75912",
    "VoMinhHuy_Heart Failure Prediction_before.csv": "d14cfc3e65d5b7eb772bec919bef749de338ea7f63a509ee66c830dd3293e8c5"
  },
  "runti

## Từ điển dữ liệu
Khoảng min-max là giá trị quan sát trong original, không phải ngưỡng sinh lý. Nguồn mô tả: https://archive.ics.uci.edu/dataset/519/heart+failure+clinical+records

In [2]:
dictionary = pd.read_csv(audit.RESULTS / 'tu_dien_du_lieu.csv')
print(dictionary[['column','meaning','unit','logical_type']].to_string(index=False))

                  column                                meaning                        unit  logical_type
                     age                         Tuổi bệnh nhân                         năm       Số thực
                 anaemia                      Chỉ báo thiếu máu                         0/1      Nhị phân
creatinine_phosphokinase           Nồng độ enzyme CPK trong máu                       mcg/L     Số nguyên
                diabetes                 Chỉ báo đái tháo đường                         0/1      Nhị phân
       ejection_fraction                     Phân suất tống máu                           %     Số nguyên
     high_blood_pressure                  Chỉ báo tăng huyết áp                         0/1      Nhị phân
               platelets            Số lượng tiểu cầu trong máu kiloplatelets/mL (theo UCI)       Số thực
        serum_creatinine         Nồng độ creatinine huyết thanh                       mg/dL       Số thực
            serum_sodium              Nồng độ 

## Giá trị thiếu và kiểu dữ liệu
Tách dòng trống khỏi thiếu từng ô để không đếm trùng hai loại vấn đề.

In [3]:
quality = pd.read_csv(audit.RESULTS / 'thong_ke_chat_luong.csv')
print(quality.loc[quality.stage == 'before_nonempty', ['column','missing','unique_nonempty','whitespace_cells','nonnumeric_nonempty']].to_string(index=False))

                  column  missing  unique_nonempty  whitespace_cells  nonnumeric_nonempty
                     age       21               50                 0                    1
                 anaemia        0                2                 0                    0
creatinine_phosphokinase       20              198                 0                    0
                diabetes        0                2                 0                    0
       ejection_fraction        0               18                 0                    0
     high_blood_pressure        0                2                 0                    0
               platelets       20              171                 0                    0
        serum_creatinine       20               40                 0                    0
            serum_sodium       21               28                 0                    0
                     sex       20                6                25                   42
          

## Thay đổi đáng lưu ý
So sánh với original chỉ phục vụ kiểm chứng. Đặc biệt, m và F có thể được chuẩn hóa về mã hợp lệ nhưng vẫn sai so với giới tính gốc.

In [4]:
changes = pd.read_csv(audit.RESULTS / 'thay_doi_can_luu_y.csv')
print(changes.to_string(index=False))

 csv_line  record_index                   column  original_value before_value            change_type
       10             9                      sex               0            m category_value_changed
       12            11                      age              75          250  numeric_value_changed
       17            16 creatinine_phosphokinase             379        99999  numeric_value_changed
       20            19             serum_sodium             140          999  numeric_value_changed
       24            23                      sex               1         F    category_value_changed
       27            26                      age              80          -12  numeric_value_changed
       32            31        ejection_fraction              38          150  numeric_value_changed
       47            46                      age              50  Invalid_Age             wrong_type
       62            61              DEATH_EVENT               1            2         inval

## Hồ sơ sao chép
So sánh nguyên trạng tìm được 3 dòng dư. Chuẩn hóa số và cách ghi sex tìm được 5 dòng dư; cần xử lý trước khi điền thiếu.

In [5]:
print(pd.read_csv(audit.RESULTS / 'trung_lap_sau_chuan_hoa.csv').to_string(index=False))

 csv_line  first_csv_line
      301               7
      302              14
      303              30
      304              52
      305             102


## Bước tiếp theo
Xác lập quy tắc dữ liệu thiếu, giá trị sai và nhãn không hợp lệ; chạy Python; đối chiếu SQL sau. Không sao chép dữ liệu gốc để thay cho thuật toán điền thiếu. Không kết luận y khoa từ các thống kê mô tả.